# S1 — Are the brain data what we think they are?

This notebook is for *looking*. All logic lives in `src/nsd_rsa/` and the figures are
produced by `scripts/s1_sanity_checks.py`; nothing here is part of the pipeline.

**What we are checking, and why it comes before any analysis.**

A beta is one number per (cortical vertex, image presentation): how strongly that patch
of cortex responded to that picture, in percent signal change. Stack them and you get a
matrix of shape `(presentations, vertices)` — structurally identical to a batch of
network activations, which is exactly why the same RDM code will work on both.

Before trusting any of it we ask three questions:

1. **Do the numbers look like betas at all?** Percent signal change should sit near zero
   with a spread of a few percent. Wildly different values would mean we misread the
   file format or the scaling.
2. **Are the ROIs the right size?** Early visual cortex and ventral cortex should contain
   thousands of vertices each. Near-empty ROIs would mean the atlas is misaligned.
3. **Can we recover something already known?** This is the real test. Split-half
   reliability should be *higher in early visual cortex than in anterior ventral regions*.
   Early cortex tracks low-level image properties in a stimulus-locked way; anterior
   regions are modulated by attention, memory and task state, so repeated presentations
   of the same image produce less similar responses. If our pipeline gets this backwards,
   something is mislabelled and every novel result would be worthless.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from nsd_rsa.loaders import load_subject, average_repeats, split_half
from nsd_rsa.noise_ceiling import split_half_reliability
from nsd_rsa.rdm import compute_rdm

files = sorted((ROOT / "data/betas").glob("*_shared_betas.h5"))
print("subjects available:", [f.stem.split("_")[0] for f in files])
data = load_subject(files[0])
print(f"{data.subject}: betas {data.betas.shape}, {data.n_images} images")
data.roi_counts()

## 1. Do the betas look like betas?

Expect a roughly symmetric distribution centred near zero. The units are percent signal
change, so a standard deviation of a couple of percent is normal for single trials — fMRI
single-trial estimates are extremely noisy, which is the whole reason repeats exist.

In [ ]:
v = data.betas[::20].ravel()
v = v[np.isfinite(v)]
print(f"mean {v.mean():+.3f}   std {v.std():.3f}   1-99% [{np.percentile(v,1):+.2f}, {np.percentile(v,99):+.2f}]")
plt.figure(figsize=(6, 3))
plt.hist(v, bins=200, range=(-10, 10), density=True)
plt.xlabel("beta (% signal change)"); plt.ylabel("density"); plt.title("Single-trial betas")
plt.show()

## 2. Averaging over repeats

Each image was shown three times. Averaging cuts the noise standard deviation by about
`sqrt(3)` while leaving the stimulus-driven signal untouched. The result is one response
pattern per image — the object every later stage consumes.

In [ ]:
patterns, images = average_repeats(data, roi="ventral")
print(f"ventral patterns: {patterns.shape}  (images x vertices)")
print(f"std of single trials: {data.betas[:, data.roi_mask('ventral')].std():.3f}")
print(f"std after averaging : {patterns.std():.3f}")

## 3. The known-fact check: reliability across the hierarchy

Split the three repeats of each image into two disjoint halves, average within each, and
correlate them per vertex. Whatever the two halves agree on is reproducible signal;
whatever they don't is noise. This measures data quality without any model at all.

**Prediction: `early` > `midventral` > `ventral`.**

In [ ]:
rois = ["early", "midventral", "ventral", "midlateral", "lateral", "midparietal", "parietal"]
rel = {}
for roi in rois:
    a, b, _ = split_half(data, roi=roi, seed=0)
    rel[roi] = float(np.nanmean(split_half_reliability(a, b)))
    print(f"{roi:<12} r = {rel[roi]:+.3f}   ({data.roi_counts()[roi]:>6,} vertices)")

print()
print("early > ventral ?", rel["early"] > rel["ventral"])

In [ ]:
plt.figure(figsize=(6, 3.2))
plt.bar(range(len(rois)), [rel[r] for r in rois])
plt.xticks(range(len(rois)), rois, rotation=20, ha="right")
plt.ylabel("split-half reliability (r)"); plt.axhline(0, color="k", lw=0.5)
plt.title(f"{data.subject}: reproducible signal by ROI")
plt.tight_layout(); plt.show()

## 4. A first look at a brain RDM

The object RSA actually compares: pairwise dissimilarity between the response patterns
evoked by different images. Its size depends only on the number of images, never on the
number of vertices — which is precisely what lets us compare a 4000-vertex ROI with a
768-dimensional model layer.

In [ ]:
from scipy.spatial.distance import squareform

sub = patterns[:200]
rdm = squareform(compute_rdm(sub, metric="correlation"))
plt.figure(figsize=(5, 4.2))
plt.imshow(rdm, cmap="viridis")
plt.colorbar(label="1 - Pearson r")
plt.title("Ventral RDM (first 200 images)")
plt.xlabel("image"); plt.ylabel("image")
plt.tight_layout(); plt.show()